# RNN 训练

- 连续时间 firing-rate RNN: $\begin{aligned}\dot{\vec{x}} = \frac{1}{\tau}\left\{-\vec{x}(t) + \tanh{\left[\mathbf{W}\vec{x}(t) + \mathbf{W}_{\mathrm{in}}\vec{u}(t) + \vec{b}\right]}\right\} = \vec{f}(\vec{x},\vec{u})\end{aligned}$
    - 构造 rank-$K$ recurrent weight matrix: $\begin{aligned}\mathbf{W} = \sum_{k=1}^{K}\vec{m}^{(k)}\vec{n}^{(k)\top} = \begin{bmatrix} \vec{m}^{(1)} & \ldots & \vec{m}^{(K)}\end{bmatrix} \begin{bmatrix} \vec{n}^{(1)\top} \\ \vdots \\ \vec{n}^{(K)\top}\end{bmatrix} = \mathbf{M}_{N\times K}\mathbf{N}^\top_{K\times N}\end{aligned}$
    - latent variables
        - $\begin{aligned}\vec{\kappa}(t) = \mathbf{N}^{\top}\vec{x}(t) = \begin{bmatrix} \vec{n}^{(1)\top} \\ \vdots \\ \vec{n}^{(K)\top}\end{bmatrix} \vec{x}(t) = \begin{bmatrix} \vec{n}^{(1)\top}\vec{x} \\ \ldots \\ \vec{n}^{(K)\top}\vec{x} \end{bmatrix}\end{aligned}$
        - $\mathbf{W}\vec{x} = \mathbf{M}\mathbf{N}^{\top}\vec{x} = \mathbf{M}\vec{\kappa}$

# Back Propagation Through Time (BPTT)

1. 展开 RNN 为时间序列: 

    令时间步长 $\Delta t$, 则约化 $\begin{aligned}\delta_{t} \equiv \frac{\Delta t}{\tau}\end{aligned}$

    $$
    \begin{aligned}
    \tau\frac{\vec{x}_{t+1}-\vec{x}_{t}}{\Delta t} &= - \vec{x}_{t} + \tanh{(\mathbf{W}\vec{x}_{t} + \mathbf{W}_{\mathrm{in}}\vec{u}_{t} + \vec{b})}\\
    \Rightarrow \vec{x}_{t+1} &= (1-\delta_{t})\vec{x}_{t} + \delta_{t}\tanh{(\vec{z}_{t})},\quad \vec{z}_{t} = \mathbf{W}\vec{x}_{t} + \mathbf{W}_{\mathrm{in}}\vec{u}_{t} + \vec{b}
    \end{aligned}
    $$

    这对应于一个 Euler forward $\vec{x}_{t+1} = \vec{x}_{t} + \vec{f}(\vec{x}_{t})\Delta t = \vec{F}_{\Delta t}(\vec{x}_{t})$

    对于离散时间序列 $\vec{x}_{0}\to\vec{x}_{1}\to\vec{x}_{2}\to\cdots\to\vec{x}_{T}$, 每个 "$\to$" 使用同一组 $(\mathbf{M},\mathbf{N}, \mathbf{W}_{\mathrm{in}}, \vec{b})$

2. loss

    令各时刻 $t$ 的 loss 为 $l_{t}$, 总 loss 为 $\begin{aligned}L = \sum_{t=1}^{T}l_{t}\end{aligned}$

3. Jacobian

    Discrete-time transition Jacobian 定义为 $\begin{aligned}\mathbf{A}_{t}\equiv \frac{\partial \vec{x}_{t+1}}{\partial \vec{x}_{t}}\end{aligned}$, 其也是 chain rule 中的一个元素

    在 1. 中处理为离散动力学 $\vec{x}_{t+1} = (1-\delta_{t})\vec{x}_{t} + \delta_{t}\tanh{(\vec{z}_{t})}$, 则

    $$
    \begin{aligned}
    \mathbf{A}_{t} &= \frac{\partial \vec{x}_{t+1}}{\partial \vec{x}_{t}} = \frac{\partial}{\partial \vec{x}_{t}} \left[\vec{x}_{t} + \vec{f}(\vec{x}_{t}) \Delta t\right] = \mathbf{I} + \frac{\partial\vec{f}}{\partial \vec{x}_{t}}\Delta t = \mathbf{I} + \mathbf{J}_{t}\Delta t\\
    &= \mathbf{I} + \frac{1}{\tau}(-\mathbf{I} + \mathbf{D}_{t}\mathbf{W})\Delta t = \left(1-\frac{\Delta t}{\tau}\right)\mathbf{I} + \frac{\Delta t}{\tau}\mathbf{D}_{t}\mathbf{W}\\
    &= \left(1-\delta_{t}\right)\mathbf{I} + \delta_{t}\mathbf{D}_{t}\mathbf{W}
    \end{aligned}
    $$

    > Continuous-time Jacobian $\begin{aligned}\mathbf{J} = \frac{\partial\dot{\vec{x}}}{\partial \vec{x}} = \frac{1}{\tau}(-\mathbf{I} + \mathbf{D}\mathbf{W})\end{aligned}$, 其中 $\begin{aligned}\mathbf{D} = \mathrm{diag}[\tanh^{\prime}(\vec{z})] = \mathrm{diag}[1-\tanh^2(\vec{z})]\end{aligned}$

4. gradient

    定义 $t$ 时刻的 state gradient $\begin{aligned}\vec{\lambda}_{t}\equiv\frac{\partial L}{\partial\vec{x}_{t}}\end{aligned}$

    由于时间序列 $\vec{x}_{t}\to\vec{x}_{t+1}\to\cdots\to\vec{x}_{T}$ 对应 $l_{t}\to l_{t+1}\to\cdots\to l_{T}$, 所以 $\vec{x}_{t}$ 有两种 loss distribution: 
        
    1. $\vec{x}_{t}\to l_{t}$
    2. $\vec{x}_{t}\to\vec{x}_{t+1}\to l_{t+1},l_{t+2},\ldots,l_{T}$

    因此将剩余 loss 分解为 $\begin{aligned}L_{t} = \sum_{s=t}^{T}l_{s} = l_{t}(\vec{x}_{t}) + L_{t+1}(\vec{x}_{t+1})\end{aligned}$, 其中 $\begin{aligned}L_{t+1} = \sum_{s=t+1}^{T}l_{s}\end{aligned}$

    loss state gradient $\begin{aligned}\frac{\partial L}{\partial \vec{x}_{t}} = \frac{\partial l_{t}}{\partial \vec{x}_{t}} + \frac{\partial L_{t+1}}{\partial \vec{x}_{t}} = \frac{\partial l_{t}}{\partial \vec{x}_{t}} + \left(\frac{\partial\vec{x}_{t+1}}{\partial\vec{x}_{t}}\right)^{\top}\frac{\partial L_{t+1}}{\partial\vec{x}_{t+1}}\end{aligned}$

    代入 $\vec{\lambda}_{t}$ 定义, 得到 $\begin{aligned}\vec{\lambda}_{t} = \frac{\partial l_{t}}{\partial\vec{x}_{t}} + \mathbf{A}_{t}^{\top}\vec{\lambda}_{t+1}\end{aligned}$

    于是 $\begin{aligned}\vec{\lambda}_{t+1} = \frac{\partial l_{t+1}}{\partial\vec{x}_{t+1}} + \mathbf{A}_{t+1}^{\top}\vec{\lambda}_{t+2}\end{aligned}$, 将上述迭代展开代入至同一式子中: 

    $$
    \begin{aligned}
    \vec{\lambda}_{t} = \frac{\partial l_{t}}{\partial\vec{x}_{t}} + \mathbf{A}_{t}^{\top}\frac{\partial l_{t+1}}{\partial\vec{x}_{t+1}} + \mathbf{A}_{t}^{\top}\mathbf{A}_{t+1}^{\top}\frac{\partial l_{t+2}}{\partial\vec{x}_{t+2}} + \ldots + \mathbf{A}_{t}^{\top}\mathbf{A}_{t+1}^{\top}\cdots\mathbf{A}_{T-1}^{\top}\frac{\partial l_{T}}{\partial\vec{x}_{T}}
    \end{aligned}
    $$

    这样的 discrete-time Jacobian 连乘, 根据 eigenvalue/singular value 可以判断 gradient vanishing/exploding

5. recurrent weight gradient

    - 动力学 $\begin{cases}
        \vec{x}_{t+1} &= (1-\delta_{t})\vec{x}_{t} + \delta_{t}\tanh{(\vec{z}_{t})}\\
        \vec{z}_{t} &= \mathbf{W}\vec{x}_{t} + \mathbf{W}_{\mathrm{in}}\vec{u}_{t} + \vec{b}
        \end{cases}$

    - 分量形式 $\begin{cases}x_{t+1}^{i} = (1-\delta_{t})x_{t}^{i} + \delta_{t}\tanh{z_{t}^{i}}\\ z_{t}^{i} = \sum_{j} W_{ij} x_{t}^{j} + \sum_{k} W_{\mathrm{in},ik} u_{t}^{k} + b_i \end{cases}$
        1. $\begin{aligned}\frac{\partial z_{t}^{i}}{\partial W_{ab}} = \delta_{ia}x_{t}^{b}\end{aligned}$
        2. $\begin{aligned}\frac{\partial x_{t+1}^{i}}{\partial W_{ab}} = \delta_{t}\tanh^{\prime}(z_{t}^{i})\frac{\partial z_{t}^{i}}{\partial W_{ab}} = \delta_{t} [1-\tanh^2(z_{t}^{i})]\delta_{ia}x_{t}^{b} = \delta_{t}D_{t}^{ii}\delta_{ia}x_{t}^{b}\end{aligned}$

    - loss gradient $\begin{aligned}\frac{\partial L}{\partial W_{ab}}\bigg|_{t} = \sum_{i}\frac{\partial L}{\partial x_{t+1}^{i}}\frac{\partial x_{t+1}^{i}}{\partial w_{ab}} = \sum_{i}\lambda_{t+1}^{i}\delta_{t}D_{t}^{ii}\delta_{ia}x_{t}^{b} = \delta_{t} D_{t}^{aa}\lambda_{t+1}^{a}x_{t}^{b}\end{aligned}$

    - 定义 error 矢量 $\vec{e}_{t} = \delta_{t}\mathbf{D}_{t}\vec{\lambda}_{t+1}$, 则 $\begin{aligned}\frac{\partial L}{\partial W_{ab}}\bigg|_{t} = e_{t}^{a}x_{t}^{b}\end{aligned}$, 张量形式 $\begin{aligned}\nabla_{\mathbf{W}}L\bigg|_{t} = \delta_{t}(\mathbf{D}_{t}\vec{\lambda}_{t+1})\vec{x}_{t}^{\top}\end{aligned}$

    - 对时刻 $t$ 求和: $\begin{aligned}\mathbf{G}_{W}\equiv \nabla_{\mathbf{W}}L = \sum_{t=0}^{T-1}\delta_{t}(\mathbf{D}_{t}\vec{\lambda}_{t+1})\vec{x}_{t}^{\top}\end{aligned}$


# Low-rank recurrent weight gradient

1. 构造 low rank recurrent weight $\begin{aligned}\mathbf{W} = \mathbf{M}\mathbf{N}^{\top} = \sum_{k=1}^{K}\vec{m}^{(k)}\vec{n}^{(k)\top}\end{aligned}$

2. 权重微分形式 $\mathrm{d}\mathbf{W} = (\mathrm{d}\mathbf{M})\mathbf{N}^{\top} + \mathbf{M}(\mathrm{d}\mathbf{N})^{\top}$

3. loss 微分形式写作 Frobenius inner product: $\mathrm{d}L = \langle \nabla_{\mathbf{W}}L, \mathrm{d}\mathbf{W}\rangle_{F} = \mathrm{Tr}[\mathbf{G}_{W}^{\top}\mathrm{d}\mathbf{W}]$
    
    代入 2. 得到 
    
    $$
    \begin{aligned}
    \mathrm{d}L &= \mathrm{Tr}[\mathbf{G}_{W}^{\top}(\mathrm{d}\mathbf{M})\mathbf{N}^{\top}] + \mathrm{Tr}[\mathbf{G}_{W}^{\top}\mathbf{M}(\mathrm{d}\mathbf{N})^{\top}]\\
    &= \mathrm{Tr}[\mathbf{N}^{\top}\mathbf{G}_{W}^{\top}\mathrm{d}\mathbf{M}] + \mathrm{Tr}[(\mathrm{d}\mathbf{N})^{\top}\mathbf{G}_{W}^{\top}\mathbf{M}]\\
    &= \mathrm{Tr}[(\mathbf{G}_{W}\mathbf{N})^{\top}\mathrm{d}\mathbf{M}] + \mathrm{Tr}[(\mathbf{G}_{W}^{\top}\mathbf{M})^{\top}(\mathrm{d}\mathbf{N})]\\
    &= \mathrm{Tr}[(\nabla_{\mathbf{M}}L)^{\top}\mathrm{d}\mathbf{M}] + \mathrm{Tr}[(\nabla_{\mathbf{N}}L)^{\top}(\mathrm{d}\mathbf{N})]\\
    \end{aligned}
    $$

    > Trace 的轮换对称: $\mathrm{Tr}[ABC] = \mathrm{Tr}[CAB]$

# Adam Optimization

BPTT 产生了对 epoch $e$ 的 gradient $\vec{g}_{e} = \nabla_{\vec{\theta}}L_{e}$

传统的 gradient descent 是 $\vec{\theta}_{e+1} = \vec{\theta}_{e} - \alpha \vec{g}_{e}$

Adam 则是通过两变量来进行 gradient 的历史记忆和更新

$$
\vec{p}_{e} = \beta_{1} \vec{p}_{e-1} + (1-\beta_{1}) \vec{g}_{e}\\
\vec{q}_{e} = \beta_{2} \vec{q}_{e-1} + (1-\beta_{2}) \vec{g}_{e}^{\odot 2}\\
\vec{p}_{0} = \vec{q}_{0} = \vec{0};\quad \beta_{1} = 0.9, \beta_{2} = 0.999\\
\hat{\vec{p}}_{e} = \frac{\vec{p}_{e}}{1-\beta_{1}^{e}}\\
\hat{\vec{q}}_{e} = \frac{\vec{q}_{e}}{1-\beta_{2}^{e}}
$$

参数更新则是 $\begin{aligned}\vec{\theta}_{e+1} = \vec{\theta}_{e} - \alpha \frac{\hat{\vec{p}}_{e}}{\sqrt{\hat{\vec{q}}_{e}} + \epsilon}\end{aligned}$ ($\alpha = 10^{-3}$)

# Low-rank recurrent weight update

对于矩阵 $\mathbf{M}$ 通过 BPTT 和 chain rule 得到 $\mathbf{G}_{M}^{e} = \nabla_{\mathbf{M}}L_e$, Adam 为其匹配 $\mathbf{P}_{e}^{M}$ 和 $\mathbf{Q}_{e}^{M}$, gate 为

$$
\begin{aligned}
\mathbf{P}_{e}^{M} &= \beta_{1}\mathbf{P}_{e-1}^{M} + (1-\beta_{1})\mathbf{G}_{M}^{e},\quad \mathbf{Q}_{e}^{M} = \beta_{2}\mathbf{Q}_{e-1}^{M} + (1-\beta_{2})(\mathbf{G}_{M}^{e})^{\odot 2}\\
\hat{\mathbf{P}}_{e}^{M} &= \frac{\mathbf{P}_{e}^{M}}{1-\beta_{1}^{e}},\quad \hat{\mathbf{Q}}_{e}^{M} = \frac{\mathbf{Q}_{e}^{M}}{1-\beta_{2}^{e}}\\
\mathbf{M}_{e+1} &= \mathbf{M}_{e} - \alpha \frac{\hat{\mathbf{P}}_{e}^{M}}{\sqrt{\hat{\mathbf{Q}}_{e}^{M}} + \epsilon}
\end{aligned}
$$

同理, 对矩阵 $\mathbf{N}$ 有

$$
\begin{aligned}
\mathbf{P}_{e}^{N} &= \beta_{1}\mathbf{P}_{e-1}^{N} + (1-\beta_{1})\mathbf{G}_{N}^{e},\quad \mathbf{Q}_{e}^{N} = \beta_{2}\mathbf{Q}_{e-1}^{N} + (1-\beta_{2})(\mathbf{G}_{N}^{e})^{\odot 2}\\
\hat{\mathbf{P}}_{e}^{N} &= \frac{\mathbf{P}_{e}^{N}}{1-\beta_{1}^{e}},\quad \hat{\mathbf{Q}}_{e}^{N} = \frac{\mathbf{Q}_{e}^{N}}{1-\beta_{2}^{e}}\\
\mathbf{N}_{e+1} &= \mathbf{N}_{e} - \alpha \frac{\hat{\mathbf{P}}_{e}^{N}}{\sqrt{\hat{\mathbf{Q}}_{e}^{N}} + \epsilon}
\end{aligned}
$$

# rank-$K$ constraint

Adam 更新的 $\mathbf{M}_{e}\to \mathbf{M}_{e+1}\in\mathbb{R}^{N\times K}$ 和 $\mathbf{N}_{e}\to \mathbf{N}_{e+1}\in\mathbb{R}^{N\times K}$, 则 $\mathbf{W}_{e+1} = \mathbf{M}_{e+1} \mathbf{N}_{e+1}^\top$

因为 $\mathrm{rank}(\mathbf{AB})\leq \min(\mathrm{rank}(\mathbf{A}), \mathrm{rank}(\mathbf{B}))$, 所以 $\mathrm{rank}(\mathbf{W}_{e+1}) \leq \min(\mathrm{rank}(\mathbf{M}_{e+1}), \mathrm{rank}(\mathbf{N}_{e+1})) \leq K$

而直接将 $\mathbf{W}$ 初始化为 $N\times N$ 的独立参数, 将不会具有这样的效果